# LSC threshold selection

Select the ACDC threshold tau* per model from the Pareto sweep results. Each model is
reviewed individually and the chosen thresholds are exported to `threshold_summary.json`
for the next stage.

KL is the divergence between the circuit and the full model on held-out data; lower is
better. KL >= 1.0 disqualifies a model. Expected circuit fractions scale with model
capacity: smaller models tolerate larger fractions.

---
## 1. Setup

In [1]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from collections import OrderedDict
from typing import Dict, List, Any, Optional, Tuple

import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 200)
pd.set_option(
    "display.float_format", lambda x: f"{x:.5f}" if abs(x) < 1 else f"{x:.2f}"
)

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 100

In [2]:
# Paths - ABSOLUTE paths (kernel may run in scratch, but data is here)
LSC_CIRCUITS_DIR = Path("LSC_circuits")

SWEEP_DIR = LSC_CIRCUITS_DIR / "pareto_sweep"
OUTPUT_DIR = SWEEP_DIR  # threshold_summary.json goes here
PLOTS_DIR = SWEEP_DIR / "threshold_selection_plots"

print(f"LSC_circuits dir: {LSC_CIRCUITS_DIR}")
print(f"Sweep directory:  {SWEEP_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Plots directory:  {PLOTS_DIR}")

LSC_circuits dir: LSC_circuits
Sweep directory:  LSC_circuits/pareto_sweep
Output directory: LSC_circuits/pareto_sweep
Plots directory:  LSC_circuits/pareto_sweep/threshold_selection_plots


---
## 2. Helper Functions

In [ ]:
# Expected circuit fractions by model capacity
EXPECTED_CIRCUIT_FRACTION = {
    # model: (min_fraction, max_fraction, typical_fraction)
    "pythia-70m": (0.15, 0.35, 0.25),
    "pythia-160m": (0.06, 0.15, 0.10),
    "pythia-410m": (0.03, 0.08, 0.05),
    "pythia-1b": (0.02, 0.05, 0.03),
    "pythia-1.4b": (0.015, 0.04, 0.025),
}

ANOMALOUS_THRESHOLDS = {}

# Max KL to consider a model viable (all thresholds above this -> model excluded)
MODEL_VIABILITY_KL = 1.0


def get_expected_fraction(model: str) -> Tuple[float, float, float]:
    """Return (min, max, typical) expected circuit fraction."""
    return EXPECTED_CIRCUIT_FRACTION.get(model, (0.05, 0.20, 0.10))


def is_anomalous(model: str, threshold: float, tol: float = 0.02) -> bool:
    """Check if a (model, threshold) pair is a known anomaly."""
    bad_thresholds = ANOMALOUS_THRESHOLDS.get(model, set())
    for bt in bad_thresholds:
        if abs(threshold - bt) / bt < tol:
            return True
    return False


def filter_anomalous(model: str, sweep_points: list) -> list:
    """Remove known anomalous thresholds from sweep points."""
    return [p for p in sweep_points if not is_anomalous(model, p["threshold"])]


def check_model_viability(model: str, sweep_points: list) -> Tuple[bool, str]:
    """Check if a model has at least one viable threshold (KL < MODEL_VIABILITY_KL)."""
    clean_points = filter_anomalous(model, sweep_points)
    if not clean_points:
        return False, "No sweep points after filtering anomalies"
    best_kl = min(p["kl_div"] for p in clean_points)
    if best_kl >= MODEL_VIABILITY_KL:
        return (
            False,
            f"Best KL={best_kl:.2f} (all >= {MODEL_VIABILITY_KL}): ACDC failed for this model",
        )
    return True, "OK"


def interpret_kl(kl: float) -> str:
    """Human-readable KL interpretation."""
    if kl < 0.1:
        return "good"
    elif kl < 0.3:
        return "moderate"
    elif kl < 0.5:
        return "acceptable"
    elif kl < 1.0:
        return "poor"
    else:
        return "bad"


def assess_circuit_size(model: str, fraction: float) -> str:
    """Assess if circuit size is appropriate."""
    min_f, max_f, _ = get_expected_fraction(model)
    if fraction < min_f:
        return "too_small"
    elif fraction <= max_f:
        return "good"
    elif fraction <= max_f * 2:
        return "large"
    else:
        return "too_large"

In [ ]:
def compute_recommendations(pareto_data: dict, model: str) -> Dict[str, Any]:
    """
    Compute recommendation heuristics.
    These are SUGGESTIONS only.

    Filters out known anomalous thresholds and enforces KL < 1.0 ceiling.
    """
    sweep_points = pareto_data.get("sweep_points", [])
    base_accuracy = pareto_data.get("base_accuracy", 0)
    if not sweep_points:
        return {}

    # Filter out anomalous data points
    clean_points = filter_anomalous(model, sweep_points)
    if not clean_points:
        return {}

    sorted_by_size = sorted(clean_points, key=lambda p: p["size_fraction"])
    recommendations = {}
    min_f, max_f, typical_f = get_expected_fraction(model)

    # KL ceiling; never recommend a point with KL >= this
    MAX_REC_KL = 1.0

    # 1. Minimal acceptable: smallest with retention >= 80% AND KL < 0.5
    for pt in sorted_by_size:
        retention = pt.get(
            "retention", pt["accuracy"] / base_accuracy if base_accuracy > 0 else 0
        )
        if retention >= 0.80 and pt["kl_div"] < 0.5:
            recommendations["minimal_acceptable"] = {
                "threshold": pt["threshold"],
                "size_fraction": pt["size_fraction"],
                "n_edges": pt["n_edges"],
                "kl_div": pt["kl_div"],
                "accuracy": pt["accuracy"],
                "retention": retention,
                "reason": "Smallest circuit with >=80% retention and KL<0.5",
            }
            break

    # 2. Interpretable: within expected size range, best retention, KL < ceiling
    in_range = [
        p
        for p in sorted_by_size
        if min_f <= p["size_fraction"] <= max_f and p["kl_div"] < MAX_REC_KL
    ]
    if in_range:
        best_in_range = max(
            in_range,
            key=lambda p: p.get(
                "retention", p["accuracy"] / base_accuracy if base_accuracy > 0 else 0
            ),
        )
        retention = best_in_range.get(
            "retention",
            best_in_range["accuracy"] / base_accuracy if base_accuracy > 0 else 0,
        )
        recommendations["interpretable"] = {
            "threshold": best_in_range["threshold"],
            "size_fraction": best_in_range["size_fraction"],
            "n_edges": best_in_range["n_edges"],
            "kl_div": best_in_range["kl_div"],
            "accuracy": best_in_range["accuracy"],
            "retention": retention,
            "reason": f"Best retention within expected range ({min_f * 100:.0f}-{max_f * 100:.0f}%) with KL<{MAX_REC_KL}",
        }

    # 3. Standard threshold τ ≈ 0.00158
    standard_tau = 0.00158
    closest = min(sorted_by_size, key=lambda p: abs(p["threshold"] - standard_tau))
    if (
        abs(closest["threshold"] - standard_tau) / standard_tau < 0.5
        and closest["kl_div"] < MAX_REC_KL
    ):
        retention = closest.get(
            "retention", closest["accuracy"] / base_accuracy if base_accuracy > 0 else 0
        )
        recommendations["standard_tau"] = {
            "threshold": closest["threshold"],
            "size_fraction": closest["size_fraction"],
            "n_edges": closest["n_edges"],
            "kl_div": closest["kl_div"],
            "accuracy": closest["accuracy"],
            "retention": retention,
            "reason": f"τ≈0.00158 (standard threshold)",
        }

    return recommendations


def get_primary_recommendation(recommendations: dict) -> Optional[dict]:
    """Get primary recommendation (priority order)."""
    for key in ["interpretable", "minimal_acceptable", "standard_tau"]:
        if key in recommendations:
            return recommendations[key]
    return None

---
## 3. Load Pareto Results

In [5]:
# Load pareto_summary.json
summary_path = SWEEP_DIR / "sweep_results" / "pareto_summary.json"

if not summary_path.exists():
    raise FileNotFoundError(
        f"pareto_summary.json not found at {summary_path}\nRun lsc_pareto_sweep.py first."
    )

with open(summary_path) as f:
    pareto_summary = json.load(f)

pareto_results = pareto_summary.get("pareto_results", {})
models = list(pareto_results.keys())

print(f"Loaded {len(models)} models: {models}")

Loaded 5 models: ['pythia-70m', 'pythia-160m', 'pythia-410m', 'pythia-1b', 'pythia-1.4b']


In [6]:
# Check model viability and compute recommendations
all_recommendations = {}
viable_models = []
excluded_models = {}

for model in models:
    pareto_data = pareto_results[model]
    sweep_points = pareto_data.get("sweep_points", [])

    # Check viability
    viable, reason = check_model_viability(model, sweep_points)
    if not viable:
        excluded_models[model] = reason
        print(f"  EXCLUDED: {model} - {reason}")
    else:
        viable_models.append(model)
        n_anomalous = len(sweep_points) - len(filter_anomalous(model, sweep_points))
        if n_anomalous > 0:
            print(f"  {model}: {n_anomalous} anomalous threshold(s) filtered out")

    # Compute recommendations (filter_anomalous is applied inside)
    all_recommendations[model] = compute_recommendations(pareto_results[model], model)

print(f"\nViable models ({len(viable_models)}): {viable_models}")
if excluded_models:
    print(f"Excluded models ({len(excluded_models)}): {list(excluded_models.keys())}")

# ============================================================================
# THRESHOLD SELECTIONS; edit these to change the highlighted rows in tables
# ============================================================================
# Selection principle: smallest circuit within the expected size range for each
# model's capacity. Larger models (1b, 1.4b) tolerate KL~0.5 for compactness.
SELECTIONS = {
    "pythia-70m": 0.00158,  # 436 edges (33.0%), KL=0.238, ret=85%, in expected range
    "pythia-160m": 0.000631,  # 1396 edges (12.2%), KL=0.284, ret=96%, in expected range
    "pythia-410m": 0.000251,  # 3444 edges (4.3%),  KL=0.285, ret=97%, in expected range
    "pythia-1b": 0.00158,  # 939 edges (9.4%),   KL=0.485, ret=94%, above range, best viable
    "pythia-1.4b": 0.000631,  # 2097 edges (2.6%),  KL=0.498, ret=94%, in expected range (1.5-4%)
}


Viable models (5): ['pythia-70m', 'pythia-160m', 'pythia-410m', 'pythia-1b', 'pythia-1.4b']


---
## 4. Model Analysis

Review each model's Pareto frontier, then make your threshold selection in Section 5.

In [7]:
def display_model_analysis(model: str):
    """Display Pareto plots for a single model."""
    pareto_data = pareto_results[model]
    sweep_points = pareto_data.get("sweep_points", [])
    base_acc = pareto_data.get("base_accuracy", 0)
    recs = all_recommendations.get(model, {})
    primary = get_primary_recommendation(recs)
    min_f, max_f, typical_f = get_expected_fraction(model)
    is_excluded = model in excluded_models

    clean_points = filter_anomalous(model, sweep_points)

    # Plot; use only clean (non-anomalous) points for curves
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left: Pareto frontier
    ax = axes[0]

    # Expected range
    ax.axvspan(
        min_f * 100, max_f * 100, alpha=0.2, color="green", label="Expected range"
    )

    # KL thresholds
    ax.axhline(0.1, color="green", ls="--", alpha=0.5, lw=1)
    ax.axhline(0.5, color="orange", ls="--", alpha=0.5, lw=1)

    # Clean points
    clean_sizes = [p["size_fraction"] * 100 for p in clean_points]
    clean_kls = [p["kl_div"] for p in clean_points]
    ax.scatter(
        clean_sizes, clean_kls, c="lightgray", s=60, zorder=2, label="All (clean)"
    )

    # Anomalous points (dimmed, distinct marker)
    anom_points = [p for p in sweep_points if is_anomalous(model, p["threshold"])]
    if anom_points:
        anom_sizes = [p["size_fraction"] * 100 for p in anom_points]
        anom_kls = [p["kl_div"] for p in anom_points]
        ax.scatter(
            anom_sizes,
            anom_kls,
            c="none",
            edgecolors="red",
            s=60,
            marker="x",
            zorder=2,
            linewidths=1.5,
            label="Anomaly (filtered)",
        )

    # Pareto points (from clean only)
    pareto_clean = [p for p in clean_points if p.get("is_pareto_optimal", False)]
    if pareto_clean:
        p_sizes = [p["size_fraction"] * 100 for p in pareto_clean]
        p_kls = [p["kl_div"] for p in pareto_clean]
        sorted_pareto = sorted(zip(p_sizes, p_kls))
        ax.plot(
            [x[0] for x in sorted_pareto],
            [x[1] for x in sorted_pareto],
            "ro-",
            markersize=10,
            zorder=3,
            label="Pareto",
        )

    # Selected threshold
    sel_tau = SELECTIONS.get(model)
    if sel_tau:
        sel_pt = None
        for p in clean_points:
            if abs(p["threshold"] - sel_tau) / p["threshold"] < 0.02:
                sel_pt = p
                break
        if sel_pt:
            ax.scatter(
                [sel_pt["size_fraction"] * 100],
                [sel_pt["kl_div"]],
                c="blue",
                s=250,
                marker="*",
                zorder=4,
                label="Selected",
                edgecolors="black",
                linewidths=1.5,
            )

    ax.set_xlabel("Circuit Size (%)")
    ax.set_ylabel("KL Divergence")
    title_suffix = " [EXCLUDED]" if is_excluded else ""
    ax.set_title(f"{model}: Pareto Frontier{title_suffix}")
    ax.legend(loc="upper right")
    ax.grid(True, alpha=0.3)

    # Right: Accuracy comparison (clean points only)
    ax = axes[1]
    sorted_clean = sorted(clean_points, key=lambda x: x["n_edges"])
    edges = [p["n_edges"] for p in sorted_clean]
    accs = [p["accuracy"] * 100 for p in sorted_clean]
    abls = [p.get("ablation_accuracy", 0) * 100 for p in sorted_clean]

    ax.plot(edges, accs, "o-", color="steelblue", label="Circuit acc", markersize=8)
    ax.plot(edges, abls, "^-", color="coral", label="Ablation acc", markersize=8)
    ax.axhline(
        base_acc * 100, color="green", ls="--", label=f"Base ({base_acc * 100:.1f}%)"
    )

    if sel_tau and sel_pt:
        ax.axvline(sel_pt["n_edges"], color="blue", ls=":", alpha=0.7, label="Selected")

    ax.set_xlabel("Number of Edges")
    ax.set_ylabel("Accuracy (%)")
    ax.set_title(f"{model}: Accuracy vs Circuit Size{title_suffix}")
    ax.legend(loc="lower right")
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(
        PLOTS_DIR / f"{model.replace('-', '_')}_analysis.png",
        dpi=150,
        bbox_inches="tight",
    )
    plt.close()

In [8]:
# Create output directory for plots
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

### 4.1 Per-model threshold tables

In [9]:
def model_table(model: str, selected_tau: Optional[float] = None):
    """Display a clean per-model threshold table with the selected row marked."""
    pareto_data = pareto_results[model]
    sweep_points = pareto_data.get("sweep_points", [])
    base_acc = pareto_data.get("base_accuracy", 0)
    min_f, max_f, typical_f = get_expected_fraction(model)
    total_edges = sweep_points[0]["total_edges"] if sweep_points else 0

    clean_points = filter_anomalous(model, sweep_points)
    n_filtered = len(sweep_points) - len(clean_points)
    sorted_pts = sorted(clean_points, key=lambda p: p["n_edges"])

    # Header
    print(f"{'=' * 90}")
    print(
        f"  {model}   |   base acc: {base_acc:.1%}   |   "
        f"total edges: {total_edges:,}   |   expected size: {min_f * 100:.0f}-{max_f * 100:.0f}%"
    )
    if n_filtered > 0:
        print(f"  ({n_filtered} anomalous threshold(s) excluded)")
    print(f"{'=' * 90}")

    rows = []
    for pt in sorted_pts:
        retention = pt.get(
            "retention", pt["accuracy"] / base_acc if base_acc > 0 else 0
        )
        size_ok = assess_circuit_size(model, pt["size_fraction"])

        # Check if this is the selected threshold
        is_selected = False
        if selected_tau is not None:
            is_selected = abs(pt["threshold"] - selected_tau) / pt["threshold"] < 0.02

        rows.append(
            {
                "": ">>>" if is_selected else "",
                "tau": f"{pt['threshold']:.2e}",
                "Edges": pt["n_edges"],
                "Size %": f"{pt['size_fraction'] * 100:.1f}",
                "KL Div": f"{pt['kl_div']:.4f}",
                "KL": interpret_kl(pt["kl_div"]),
                "Acc %": f"{pt['accuracy'] * 100:.1f}",
                "Retention %": f"{retention * 100:.1f}",
                "Size Fit": size_ok,
                "Pareto": "*" if pt.get("is_pareto_optimal") else "",
            }
        )

    df = pd.DataFrame(rows)
    display(
        df.style.hide(axis="index")
        .set_properties(**{"text-align": "right"})
        .set_properties(subset=[""], **{"text-align": "center", "font-weight": "bold"})
    )
    print()


# Display tables for all viable models
for model in viable_models:
    sel_tau = SELECTIONS.get(model)
    model_table(model, selected_tau=sel_tau)

  pythia-70m   |   base acc: 62.2%   |   total edges: 1,324   |   expected size: 15-35%


,tau,Edges,Size %,KL Div,KL,Acc %,Retention %,Size Fit,Pareto
,1.00e-02,201,15.2,0.7530,poor,36.9,59.3,good,*
,3.98e-03,295,22.3,0.3955,acceptable,48.4,77.9,good,*
>>>,1.58e-03,436,32.9,0.2381,moderate,52.9,85.0,good,*
,6.31e-04,590,44.6,0.1253,moderate,56.4,90.7,large,*
,2.51e-04,781,59.0,0.0570,good,60.9,97.9,large,*
,1.00e-04,1008,76.1,0.0183,good,62.2,100.0,too_large,*
,3.98e-05,1187,89.7,0.0041,good,61.8,99.3,too_large,*
,1.58e-05,1300,98.2,0.0003,good,62.2,100.0,too_large,*
,6.31e-06,1321,99.8,0.0000,good,62.2,100.0,too_large,*
,1.00e-06,1324,100.0,0.0000,good,62.2,100.0,too_large,*



  pythia-160m   |   base acc: 96.9%   |   total edges: 11,467   |   expected size: 6-15%


,tau,Edges,Size %,KL Div,KL,Acc %,Retention %,Size Fit,Pareto
,1.00e-02,372,3.2,0.9226,poor,71.6,73.9,too_small,*
,3.98e-03,541,4.7,0.6253,poor,84.9,87.6,too_small,*
,1.58e-03,904,7.9,0.4144,acceptable,89.3,92.2,good,*
>>>,6.31e-04,1396,12.2,0.2836,moderate,93.3,96.3,good,*
,2.51e-04,2138,18.6,0.2078,moderate,92.9,95.9,large,*
,1.00e-04,3205,27.9,0.1306,moderate,96.0,99.1,large,*
,3.98e-05,4529,39.5,0.0682,good,95.6,98.6,too_large,*
,1.58e-05,6306,55.0,0.0280,good,96.9,100.0,too_large,*
,6.31e-06,7985,69.6,0.0102,good,97.3,100.5,too_large,*
,2.51e-06,9367,81.7,0.0027,good,97.3,100.5,too_large,*



  pythia-410m   |   base acc: 98.7%   |   total edges: 80,581   |   expected size: 3-8%


,tau,Edges,Size %,KL Div,KL,Acc %,Retention %,Size Fit,Pareto
,1.00e-02,392,0.5,2.1933,bad,40.9,41.4,too_small,*
,3.98e-03,703,0.9,1.2008,bad,69.8,70.7,too_small,*
,1.58e-03,1148,1.4,0.7691,poor,83.1,84.2,too_small,*
,6.31e-04,2073,2.6,0.3939,acceptable,91.1,92.3,too_small,*
>>>,2.51e-04,3444,4.3,0.2855,moderate,95.6,96.8,good,*
,1.00e-04,5949,7.4,0.1969,moderate,98.2,99.5,good,*
,3.98e-05,9518,11.8,0.1450,moderate,97.3,98.6,large,*
,1.58e-05,15089,18.7,0.0959,good,97.3,98.6,too_large,*
,6.31e-06,21998,27.3,0.0663,good,97.8,99.1,too_large,*
,2.51e-06,30417,37.7,0.0332,good,99.1,100.5,too_large,*



  pythia-1b   |   base acc: 97.8%   |   total edges: 10,009   |   expected size: 2-5%


,tau,Edges,Size %,KL Div,KL,Acc %,Retention %,Size Fit,Pareto
,1.00e-02,312,3.1,1.3398,bad,67.1,68.6,good,*
,3.98e-03,551,5.5,0.9003,poor,82.2,84.1,large,*
>>>,1.58e-03,939,9.4,0.4849,acceptable,92.0,94.1,large,*
,6.31e-04,1425,14.2,0.3436,acceptable,93.8,95.9,too_large,*
,2.51e-04,2261,22.6,0.2055,moderate,96.0,98.2,too_large,*
,1.00e-04,3421,34.2,0.1170,moderate,98.7,100.9,too_large,*
,3.98e-05,4795,47.9,0.0568,good,97.8,100.0,too_large,*
,1.58e-05,6141,61.4,0.0312,good,97.8,100.0,too_large,*
,6.31e-06,7423,74.2,0.0103,good,98.7,100.9,too_large,*
,2.51e-06,8510,85.0,0.0026,good,98.7,100.9,too_large,*



  pythia-1.4b   |   base acc: 98.2%   |   total edges: 80,581   |   expected size: 2-4%


,tau,Edges,Size %,KL Div,KL,Acc %,Retention %,Size Fit,Pareto
,1.00e-02,412,0.5,1.6967,bad,56.0,57.0,too_small,*
,3.98e-03,690,0.9,1.0042,bad,79.1,80.5,too_small,*
,1.58e-03,1197,1.5,0.6931,poor,85.3,86.9,too_small,*
>>>,6.31e-04,2097,2.6,0.4985,acceptable,92.4,94.1,good,*
,2.51e-04,3770,4.7,0.3064,acceptable,95.6,97.3,large,*
,1.00e-04,5866,7.3,0.2367,moderate,98.7,100.5,large,*
,3.98e-05,10072,12.5,0.1786,moderate,96.4,98.2,too_large,*
,1.58e-05,16277,20.2,0.1222,moderate,97.8,99.5,too_large,*
,6.31e-06,23370,29.0,0.0771,good,98.7,100.5,too_large,*
,2.51e-06,32251,40.0,0.0444,good,98.2,100.0,too_large,*


### 4.2 Per-Model Pareto Plots

In [10]:
# Per-model Pareto plots
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

for model in viable_models:
    display_model_analysis(model)

---
## 5. Validate and Export Selections


In [ ]:
print("Current selections:")
for model, threshold in SELECTIONS.items():
    print(f"  {model}: tau = {threshold:.5f}")

if excluded_models:
    print(f"\nExcluded (not selectable): {list(excluded_models.keys())}")

Current selections:
  pythia-70m: tau = 0.00158
  pythia-160m: tau = 0.00063
  pythia-410m: tau = 0.00025
  pythia-1b: tau = 0.00158
  pythia-1.4b: tau = 0.00063


### 5.1 Validate Selections

In [12]:
# Validate and show details for your selections
def find_closest_threshold(sweep_points, threshold, tolerance=0.02):
    """Find matching threshold with relative tolerance (default 2%)."""
    for p in sweep_points:
        if abs(p["threshold"] - threshold) / p["threshold"] < tolerance:
            return p
    return None


if SELECTIONS:
    print("VALIDATION OF YOUR SELECTIONS")
    print("=" * 80)

    validation_rows = []
    corrected_selections = {}

    for model, threshold in SELECTIONS.items():
        if model in excluded_models:
            print(
                f"WARNING: {model} is EXCLUDED ({excluded_models[model]}); removing from selections!"
            )
            continue

        if model not in pareto_results:
            print(f"WARNING: {model} not in Pareto results!")
            continue

        pareto_data = pareto_results[model]
        sweep_points = pareto_data.get("sweep_points", [])
        base_acc = pareto_data.get("base_accuracy", 0)
        min_f, max_f, _ = get_expected_fraction(model)

        # Only search in clean (non-anomalous) points
        clean_points = filter_anomalous(model, sweep_points)
        pt = find_closest_threshold(clean_points, threshold)

        if pt:
            # Auto-correct to exact sweep value
            if pt["threshold"] != threshold:
                print(
                    f"NOTE: {model} τ={threshold} -> corrected to {pt['threshold']} (from sweep)"
                )
            corrected_selections[model] = pt["threshold"]

            retention = pt.get(
                "retention", pt["accuracy"] / base_acc if base_acc > 0 else 0
            )
            size_ok = assess_circuit_size(model, pt["size_fraction"])
            validation_rows.append(
                {
                    "Model": model,
                    "Threshold": pt["threshold"],
                    "Edges": pt["n_edges"],
                    "Size%": pt["size_fraction"] * 100,
                    "KL": pt["kl_div"],
                    "KL?": interpret_kl(pt["kl_div"]),
                    "Ret%": retention * 100,
                    "SizeOK": f"{size_emoji(size_ok)} {size_ok}",
                    "Status": "OK"
                    if size_ok in ["good", "large"] and pt["kl_div"] < 1.0
                    else "CHECK",
                }
            )
        else:
            # Check if it matched an anomalous threshold
            pt_all = find_closest_threshold(sweep_points, threshold)
            if pt_all and is_anomalous(model, pt_all["threshold"]):
                print(
                    f"WARNING: {model} τ={threshold} matches an ANOMALOUS threshold; pick a different one!"
                )
            else:
                print(
                    f"WARNING: {model} τ={threshold} not found in sweep (>2% off)! Will need fresh ACDC run."
                )
            corrected_selections[model] = threshold
            validation_rows.append(
                {
                    "Model": model,
                    "Threshold": threshold,
                    "Edges": "?",
                    "Size%": "?",
                    "KL": "?",
                    "KL?": "?",
                    "Ret%": "?",
                    "SizeOK": "?",
                    "Status": "ANOMALY"
                    if (pt_all and is_anomalous(model, pt_all["threshold"]))
                    else "NEW_ACDC",
                }
            )

    if validation_rows:
        display(pd.DataFrame(validation_rows))

    # Update SELECTIONS with corrected values (excluding removed models)
    SELECTIONS = corrected_selections
    if corrected_selections != SELECTIONS:
        print()
        print("Corrected SELECTIONS (use these exact values):")
        for m, t in corrected_selections.items():
            print(f'    "{m}": {t},')
else:
    print("No selections to validate. Edit SELECTIONS in the cell above.")

VALIDATION OF YOUR SELECTIONS


,Model,Threshold,Edges,Size%,KL,KL?,Ret%,SizeOK,Status
0,pythia-70m,0.00158,436,32.93,0.23805,moderate,85.00,good,OK
1,pythia-160m,0.00063,1396,12.17,0.28356,moderate,96.33,good,OK
2,pythia-410m,0.00025,3444,4.27,0.28549,moderate,96.85,good,OK
3,pythia-1b,0.00158,939,9.38,0.48489,acceptable,94.09,~ large,OK
4,pythia-1.4b,0.00063,2097,2.60,0.49848,acceptable,94.12,good,OK


---
## 6. Export threshold_summary.json

In [13]:
def export_threshold_summary(selections: Dict[str, float], output_dir: Path):
    """Export threshold_summary.json for Phase 2."""
    output_path = output_dir / "sweep_results" / "threshold_summary.json"
    output_path.parent.mkdir(parents=True, exist_ok=True)

    # Build selections with metadata
    selections_with_meta = {}
    for model, threshold in selections.items():
        pareto_data = pareto_results.get(model, {})
        sweep_points = pareto_data.get("sweep_points", [])
        base_acc = pareto_data.get("base_accuracy", 0)

        # Find matching sweep point with tolerance
        pt = find_closest_threshold(sweep_points, threshold)

        if pt:
            retention = pt.get(
                "retention", pt["accuracy"] / base_acc if base_acc > 0 else 0
            )
            selections_with_meta[model] = {
                "threshold": pt["threshold"],  # Use exact sweep value
                "size_fraction": pt["size_fraction"],
                "n_edges": pt["n_edges"],
                "kl_div": pt["kl_div"],
                "accuracy": pt["accuracy"],
                "retention": retention,
                "ablation_accuracy": pt.get("ablation_accuracy", 0),
                "is_pareto_optimal": pt.get("is_pareto_optimal", False),
            }
        else:
            selections_with_meta[model] = {
                "threshold": threshold,
                "note": "Custom threshold (not from Pareto sweep)",
            }

    summary = OrderedDict()
    summary["created_at"] = datetime.now().isoformat()
    summary["source"] = "lsc_threshold_select.ipynb"
    summary["note"] = "Human-selected thresholds for Phase 2 circuit discovery"
    summary["selections"] = selections_with_meta

    with open(output_path, "w") as f:
        json.dump(summary, f, indent=2)

    return output_path

In [ ]:
if not SELECTIONS:
    print("ERROR: No selections to export!")
    print("Edit the SELECTIONS dict in Section 5 first.")
else:
    output_path = export_threshold_summary(SELECTIONS, OUTPUT_DIR)

    print("=" * 80)
    print("THRESHOLD SELECTION COMPLETE")
    print("=" * 80)
    print(f"\nSaved to: {output_path}")
    print(f"\nSelections (exact values exported):")

    # Read back the actual exported values to display
    with open(output_path) as f:
        exported = json.load(f)
    for model, data in exported["selections"].items():
        t = data["threshold"]
        print(f"  {model}: τ = {t}")

    print(f"\nNext step: Run lsc_acdc_circuit.py for Phase 2")
    print(f"  python lsc_acdc_circuit.py --sweep-dir {OUTPUT_DIR}")

THRESHOLD SELECTION COMPLETE

Saved to: LSC_circuits/pareto_sweep/sweep_results/threshold_summary.json

Selections (exact values exported):
  pythia-70m: τ = 0.00158
  pythia-160m: τ = 0.000631
  pythia-410m: τ = 0.000251
  pythia-1b: τ = 0.00158
  pythia-1.4b: τ = 0.000631

Next step: Run lsc_acdc_circuit.py for Phase 2
  python lsc_acdc_circuit.py --sweep-dir LSC_circuits/pareto_sweep


---
## 7. Verify Export

In [15]:
# Read back and display the exported file
export_path = OUTPUT_DIR / "sweep_results" / "threshold_summary.json"

if export_path.exists():
    with open(export_path) as f:
        exported = json.load(f)

    print("Exported threshold_summary.json:")
    print(json.dumps(exported, indent=2))
else:
    print("threshold_summary.json not found. Run the export cell above.")

Exported threshold_summary.json:
{
  "created_at": "2026-02-24T17:42:39.581472",
  "source": "lsc_threshold_select.ipynb",
  "note": "Human-selected thresholds for Phase 2 circuit discovery",
  "selections": {
    "pythia-70m": {
      "threshold": 0.00158,
      "size_fraction": 0.3293051359516616,
      "n_edges": 436,
      "kl_div": 0.23805111646652222,
      "accuracy": 0.5288888888888889,
      "retention": 0.85,
      "ablation_accuracy": 0.0,
      "is_pareto_optimal": true
    },
    "pythia-160m": {
      "threshold": 0.000631,
      "size_fraction": 0.12174064707421296,
      "n_edges": 1396,
      "kl_div": 0.28355976939201355,
      "accuracy": 0.9333333333333333,
      "retention": 0.963302752293578,
      "ablation_accuracy": 0.0,
      "is_pareto_optimal": true
    },
    "pythia-410m": {
      "threshold": 0.000251,
      "size_fraction": 0.042739603628646955,
      "n_edges": 3444,
      "kl_div": 0.285493940114975,
      "accuracy": 0.9555555555555556,
      "retenti